In [ ]:
# hide
# no-output
from IPython.utils.capture import capture_output
with capture_output():
    %pip install -q plotly anywidget

import asyncio
import os
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import Audio
import pyquist as pq
import icm_plotly
from icm_plotly import RED, BLUE, GOLD, IRON, STEEL

Drag the loop points: the top panel shows where the loop sits in the
recording, and the bottom panel zooms in on the moment the loop wraps
around, from the end of one pass (left of the dashed line) to the start of
the next (right). The gray curve is the recording itself around the loop
start, which is what a perfect wrap would look like. The audio card
underneath always plays an 8-second note with the current loop.

In [ ]:
# hide
# autorun
sax = pq.Audio.from_file("../assets/sax-C5.wav")
sr = sax.sample_rate
x = np.asarray(sax.samples).reshape(-1)
F0 = 523.25                                   # the recording is C5
DUR = 8.0                                     # the note we build from it (s)
START0, END0, XFADE0 = 3050.0, 3300.0, 0.0    # loop points and crossfade (ms)
W = int(0.006 * sr)                           # the wrap view spans +/- 6 ms
YMAX = 1.2 * np.abs(x).max()

blocks = x[:len(x) // 96 * 96].reshape(-1, 96)   # the recording, decimated for display
t_rec = np.repeat((np.arange(len(blocks)) * 96 + 48) / sr, 2)
y_rec = np.column_stack([blocks.min(1), blocks.max(1)]).reshape(-1)
t_wrap = np.arange(-W, W) / sr * 1000


def loop_points(start_ms, end_ms, xfade_ms, sr=sr, n=len(x), W=W):
    # loop start s, loop end e, and crossfade length L, all in samples
    s = int(round(start_ms / 1000 * sr))
    e = min(n - 1, max(s + W, int(round(end_ms / 1000 * sr))))
    L = min(int(round(xfade_ms / 1000 * sr)), e - s)
    return s, e, L


def loop_body(s, e, L, x=x):
    # one pass of the loop, its last L samples crossfaded into the L samples before s
    body = x[s:e].copy()
    if L > 0:
        w = np.linspace(0, 1, L)
        body[-L:] = (1 - w) * x[e - L:e] + w * x[s - L:s]
    return body


def wrap(s, e, L, x=x, W=W, loop_body=loop_body):
    # the output around a wrap: the end of one pass, then the start of the next
    return np.concatenate([loop_body(s, e, L)[-W:], x[s:s + W]])


def figure():
    s, e, L = loop_points(START0, END0, XFADE0)
    a, b = s / sr, e / sr
    fig = make_subplots(rows=2, cols=1, vertical_spacing=0.22)
    fig.add_scatter(x=[a, a, b, b, a], y=[-YMAX, YMAX, YMAX, -YMAX, -YMAX],
                    fill="toself", fillcolor="rgba(253, 181, 21, 0.35)",
                    line=dict(width=0), hoverinfo="skip", row=1, col=1)
    fig.add_scatter(x=t_rec, y=y_rec, mode="lines", line=dict(color=IRON, width=1),
                    row=1, col=1)
    fig.add_scatter(x=[0, 0], y=[-YMAX, YMAX], mode="lines",
                    line=dict(color=GOLD, width=1.5, dash="dash"), row=2, col=1)
    fig.add_scatter(x=t_wrap, y=x[s - W:s + W], mode="lines",
                    line=dict(color=STEEL, width=4), row=2, col=1)
    fig.add_scatter(x=t_wrap, y=wrap(s, e, L), mode="lines",
                    line=dict(color=RED, width=1.6), row=2, col=1)
    fig.update_xaxes(range=[0, len(x) / sr], title_text="Time in the recording (s)",
                     fixedrange=True, row=1, col=1)
    fig.update_xaxes(range=[-W / sr * 1000, W / sr * 1000],
                     title_text="Time around the wrap (ms)", fixedrange=True, row=2, col=1)
    fig.update_yaxes(range=[-YMAX, YMAX], title_text="Amplitude", fixedrange=True)
    fig.update_layout(height=460, showlegend=False)
    return fig


def controls(fig):
    start = widgets.FloatSlider(description="Loop start (ms)", min=100.0, max=4000.0,
                                step=0.1, value=START0, readout_format=".1f")
    end = widgets.FloatSlider(description="Loop end (ms)", min=150.0, max=4300.0,
                              step=0.1, value=END0, readout_format=".1f")
    xfade = widgets.FloatSlider(description="Crossfade (ms)", min=0.0, max=50.0,
                                step=0.5, value=XFADE0, readout_format=".1f")
    readout = widgets.HTML()

    # the defaults snapshot the arrays; the page's notebooks share one kernel
    def update(s_ms, e_ms, x_ms, x=x, sr=sr, W=W, F0=F0, loop_points=loop_points,
               wrap=wrap):
        s, e, L = loop_points(s_ms, e_ms, x_ms)
        with fig.batch_update():
            fig.data[0].x = [s / sr, s / sr, e / sr, e / sr, s / sr]
            fig.data[3].y = x[s - W:s + W]
            fig.data[4].y = wrap(s, e, L)
        c = F0 * (e - s) / sr                 # cycles of C5 in the loop
        readout.value = (f"<span style='font-size:0.9em'><i>c</i> = {c:.2f} cycles "
                         f"&nbsp;·&nbsp; {abs(c - round(c)):.2f} cycles from a whole "
                         f"number</span>")

    widgets.interactive_output(update, {"s_ms": start, "e_ms": end, "x_ms": xfade})

    # the audio card under the controls: the previous clip stays in place
    # while you drag (so the layout never jumps) and is swapped for the new
    # one when the pointer releases (keyboard nudges settle on a timer). It is
    # written through the Output's synced `outputs` trait, which works
    # outside a kernel message, where display() output has no destination
    out = widgets.Output()
    gate = icm_plotly.release_gate()   # pointer state: is a slider mid-drag?
    pending = []
    dirty = []

    def render(x=x, sr=sr, DUR=DUR, loop_points=loop_points, loop_body=loop_body):
        s, e, L = loop_points(start.value, end.value, xfade.value)
        n_loops = max(1, round((DUR * sr - s - (len(x) - e)) / (e - s)))
        # the final pass isn't crossfaded: it flows straight into the release
        y = np.concatenate([x[:s]] + [loop_body(s, e, L)] * (n_loops - 1) + [x[s:]])
        y *= 0.125 / np.abs(y).max()          # about -18 dBFS, a safe level
        audio = Audio(y.astype(np.float32), rate=sr, normalize=False)
        data, metadata = get_ipython().display_formatter.format(audio)
        # one assignment swaps the old card for the new one in place, so
        # the page never shows an empty card and nothing shifts
        out.outputs = ({"output_type": "display_data",
                        "data": data, "metadata": metadata},)

    async def settle():
        await asyncio.sleep(0.25)
        pending.clear()
        if dirty and not gate.dragging:
            dirty.clear()
            render()

    def on_change(_):
        dirty.append(True)
        if pending:
            pending.pop().cancel()
        pending.append(asyncio.ensure_future(settle()))

    def on_release(change):
        if not change["new"] and dirty:
            if pending:
                pending.pop().cancel()
            dirty.clear()
            render()

    gate.observe(on_release, names="dragging")

    for s in (start, end, xfade):
        s.observe(on_change, names="value")
    if not os.environ.get("ICM_BOOK_BUILD"):   # the build bakes no card
        render()
    return widgets.VBox([start, end, xfade, readout, out, gate])

icm_plotly.show(figure, controls)